# 01 Cleaning: Toronto Building Permits (2017 onward)

**Goal of this notebook:** turn the two raw City of Toronto files (Active Permits and Cleared Permits) into one clean table, `data/clean/permits_clean.csv`, with one row per permit, ready for analysis.

**Steps**
1. Load both raw files and check their columns match
2. Combine them and record which file each row came from
3. Parse dates and keep applications from 2017 onward
4. Remove duplicates and keep the latest revision per permit number
5. Clean declared construction cost
6. Derive FSA, application year and month, and approval days
7. Group permit and work types into clean categories
8. Save the clean file and print the cleaning log

Every step that drops or changes rows is recorded with `log_step()`, and the final cell prints a table to paste into `docs/cleaning_log.md`.

> Cells marked **CHECK** print something you should look at before trusting the next step. Where I could not confirm the data's behaviour in advance, the cell says so.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Paths are relative to the repo root. The notebook lives in notebooks/,
# so the root is one folder up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

SCOPE_START = pd.Timestamp("2017-01-01")

# Cleaning log: one entry per step, so every row we drop is accounted for.
cleaning_log = []

def log_step(step, rows_before, rows_after, note=""):
    cleaning_log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "note": note,
    })
    print(f"{step}: {rows_before:,} -> {rows_after:,} ({rows_before - rows_after:,} removed)")

## 1. Load the raw files

Everything is read as text (`dtype=str`) on purpose. Permit numbers and postal codes are identifiers, not numbers, and the cost column contains placeholder text in some rows. Reading as text means nothing is silently converted; we convert each column ourselves, on purpose, later on.

In [ ]:
active = pd.read_csv(RAW / "active_permits.csv", dtype=str, low_memory=False)
cleared = pd.read_csv(RAW / "cleared_permits_since_2017.csv", dtype=str, low_memory=False)

print("Active: ", active.shape)
print("Cleared:", cleared.shape)

In [ ]:
# CHECK: the two files should have the same columns. Anything printed here
# needs a decision before combining.
only_active = set(active.columns) - set(cleared.columns)
only_cleared = set(cleared.columns) - set(active.columns)
print("Only in Active: ", sorted(only_active))
print("Only in Cleared:", sorted(only_cleared))
print()
print(list(active.columns))

## 2. Combine

We tag each row with its source file before stacking, because the same permit can appear in both files (for example, a permit that was active when one snapshot was taken and cleared later). The tag lets us decide which copy to keep in step 4.

`_id` is a row number added by the portal's datastore. It means nothing across files, so we drop it.

In [ ]:
active["source"] = "active"
cleared["source"] = "cleared"

permits = pd.concat([active, cleared], ignore_index=True)
permits = permits.drop(columns=["_id"], errors="ignore")

# Trim stray spaces in every text column. Values like "M5V " and "M5V"
# would otherwise count as two different FSAs.
text_cols = permits.select_dtypes(include="object").columns
permits[text_cols] = permits[text_cols].apply(lambda s: s.str.strip())

rows_start = len(permits)
log_step("Combine Active + Cleared", len(active) + len(cleared), rows_start)

## 3. Dates and the 2017+ scope

The Cleared file is named "since 2017", but a quick look at the portal's API showed it also contains permits *applied for* long before 2017 (for example 1986 and 2005 applications). So "since 2017" seems to refer to when permits were cleared, not when they were applied for. Our scope is by **application date**, so we filter on that.

In [ ]:
for col in ["APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE"]:
    # errors="coerce" turns unparseable values into NaT so we can count them
    # instead of the notebook crashing.
    permits[col] = pd.to_datetime(permits[col], errors="coerce")

# CHECK: how many dates are missing or unparseable?
print(permits[["APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE"]].isna().sum())
print()
print("Application date range:", permits["APPLICATION_DATE"].min(), "to", permits["APPLICATION_DATE"].max())

In [ ]:
before = len(permits)
permits = permits[permits["APPLICATION_DATE"] >= SCOPE_START].copy()
log_step("Keep applications from 2017-01-01 onward", before, len(permits),
         "Rows with a missing application date are also removed here")

## 4. Duplicates and latest revision

Three kinds of repetition are possible:
- **Exact duplicate rows** (identical in every column). Safe to drop.
- **Same permit and revision in both files.** We keep the **Cleared** copy, because a permit is cleared after it has been active, so the Cleared row should hold the later state. *This is an assumption: check the overlap printed below and confirm the Cleared copy looks newer (for example, a completed date or a closed status).*
- **Several revisions of one permit.** `REVISION_NUM` is stored as text like "00", "01". We convert it to a number and keep the highest per `PERMIT_NUM`, which is the brief's rule of one row per permit at its latest revision.

In [ ]:
before = len(permits)
permits = permits.drop_duplicates(subset=[c for c in permits.columns if c != "source"])
log_step("Drop exact duplicate rows", before, len(permits))

In [ ]:
# CHECK: permits that appear in both files at the same revision.
key = ["PERMIT_NUM", "REVISION_NUM"]
in_both = permits.groupby(key)["source"].nunique()
in_both = in_both[in_both > 1].index
print(f"Permit + revision pairs in both files: {len(in_both):,}")

sample = permits.set_index(key).loc[in_both[:5]].reset_index()
sample[key + ["source", "STATUS", "ISSUED_DATE", "COMPLETED_DATE"]]

In [ ]:
permits["revision_int"] = pd.to_numeric(permits["REVISION_NUM"], errors="coerce")
print("Revision numbers that are not numeric:", permits["revision_int"].isna().sum())

# Sort so the row we want to keep comes LAST within each permit:
# highest revision, and on a tie, Cleared after Active.
permits["source_rank"] = permits["source"].map({"active": 0, "cleared": 1})
permits = permits.sort_values(["PERMIT_NUM", "revision_int", "source_rank"])

before = len(permits)
permits = permits.drop_duplicates(subset="PERMIT_NUM", keep="last")
log_step("Keep latest revision per PERMIT_NUM (Cleared wins ties)", before, len(permits))

permits = permits.drop(columns=["source_rank"])

## 5. Declared construction cost

`EST_CONST_COST` is published as text. Some rows hold a placeholder sentence instead of a number (seen in the API: "DO NOT UPDATE OR DELETE THIS INFO FIELD"). These appear to be trade permits (plumbing, HVAC, drains) attached to a main building permit, where the cost is recorded on the main permit. We convert to a number, turn non-numbers into missing, and **do not fill them in**: making up costs would distort the value analysis.

We keep these rows. They still count as permits; they are just excluded from value totals.

In [ ]:
raw_cost = permits["EST_CONST_COST"]
permits["est_cost"] = pd.to_numeric(raw_cost.str.replace(",", "", regex=False), errors="coerce")

# CHECK: what non-numeric values were there?
non_numeric = raw_cost[permits["est_cost"].isna() & raw_cost.notna()]
print("Non-numeric cost values (top 10):")
print(non_numeric.value_counts().head(10))
print()
print("Missing cost after conversion:", permits["est_cost"].isna().sum())
print("Cost exactly 0:              ", (permits["est_cost"] == 0).sum())
print("Negative cost:               ", (permits["est_cost"] < 0).sum())
permits["est_cost"].describe()

## 6. Derived columns: FSA, year, month, approval days

**FSA.** In the API sample, `POSTAL` already held only three characters (like "M5V"). Taking the first three characters works whether the file has the FSA or a full postal code. A valid Toronto FSA is the letter M, a digit, then a letter, so anything else is flagged as invalid and set to missing (the permit is kept, but it cannot be mapped).

**Approval days** = issued date minus application date. It only exists for permits that have been issued. Negative values would be data errors and are set to missing.

**Open question to check:** after step 4 each permit is at its latest revision. I do not yet know whether a revision row carries the *original* application date or the date the revision was applied for. If it is the revision date, approval days for revised permits could be wrong (even negative). The negative count printed below is the first clue; if it is large, we should take the application date from revision "00" instead.

In [ ]:
permits["fsa"] = permits["POSTAL"].str.upper().str[:3]
valid_fsa = permits["fsa"].str.fullmatch(r"M\d[A-Z]", na=False)

# CHECK: what do the invalid values look like?
print("Invalid or missing FSA:", (~valid_fsa).sum())
print(permits.loc[~valid_fsa, "POSTAL"].value_counts(dropna=False).head(10))
permits.loc[~valid_fsa, "fsa"] = np.nan

permits["app_year"] = permits["APPLICATION_DATE"].dt.year
permits["app_month"] = permits["APPLICATION_DATE"].dt.month
# First day of the month: handy for monthly trend charts in Tableau.
permits["app_year_month"] = permits["APPLICATION_DATE"].dt.to_period("M").dt.to_timestamp()

permits["approval_days"] = (permits["ISSUED_DATE"] - permits["APPLICATION_DATE"]).dt.days
neg = permits["approval_days"] < 0
print("Negative approval days (set to missing):", neg.sum())
permits.loc[neg, "approval_days"] = np.nan

permits["approval_days"].describe()

## 7. Categories

The raw `PERMIT_TYPE`, `STRUCTURE_TYPE` and `WORK` columns have many specific values. For the analysis we need two simple groupings:
- **sector**: Residential vs Non-residential (and Mixed use)
- **work_category**: New construction, Addition, Renovation / alteration, Demolition, Other

I have not seen the full list of values yet, so **first print them, then fill in the mapping**. The keyword rules below are a first draft based on value names I saw in a small sample (for example "Apartment Building", "Mixed Use/Res w Non Res", "Partial Permit - Foundation"). Anything that does not match falls into "Other" and is printed so you can decide where it belongs.

In [ ]:
# CHECK: share these counts so the mappings can be finalised.
for col in ["PERMIT_TYPE", "STRUCTURE_TYPE", "WORK"]:
    print(f"--- {col}: {permits[col].nunique()} distinct values ---")
    print(permits[col].value_counts(dropna=False).head(40).to_string())
    print()

In [ ]:
# FIRST DRAFT: keyword rules. Lower-case matching, first match wins.
# Review the "Other" output below and refine these lists.

# "SFD" = single family dwelling (seen as "SFD - Detached" style values).
RESIDENTIAL_WORDS = ["sfd", "detached", "semi", "house", "dwelling", "apartment",
                     "residential", "townhouse", "duplex", "triplex", "fourplex",
                     "multiplex", "condo", "laneway", "garage"]
MIXED_WORDS = ["mixed use"]

def map_sector(structure_type):
    s = str(structure_type).lower()
    if any(w in s for w in MIXED_WORDS):
        return "Mixed use"
    if any(w in s for w in RESIDENTIAL_WORDS):
        return "Residential"
    if s in ("nan", "", "none"):
        return "Unknown"
    return "Non-residential"

# Order matters: "demolition" is checked before "new" and so on.
WORK_RULES = [
    ("Demolition", ["demolition"]),
    ("New construction", ["new building", "new construction", "new house", "new "]),
    ("Addition", ["addition", "additional storey", "extension"]),
    ("Renovation / alteration", ["alteration", "interior", "renovat", "repair",
                                 "underpinning", "change of use", "porch", "deck"]),
]

def map_work(work):
    w = str(work).lower()
    for category, words in WORK_RULES:
        if any(word in w for word in words):
            return category
    return "Other"

permits["sector"] = permits["STRUCTURE_TYPE"].map(map_sector)
permits["work_category"] = permits["WORK"].map(map_work)

print(permits["sector"].value_counts(dropna=False))
print()
print(permits["work_category"].value_counts(dropna=False))

In [ ]:
# CHECK: the WORK values that ended up in "Other". If big groups land here,
# add their keywords to WORK_RULES above and re-run.
permits.loc[permits["work_category"] == "Other", "WORK"].value_counts().head(30)

## 8. Save and print the cleaning log

In [ ]:
keep_cols = [
    "PERMIT_NUM", "REVISION_NUM", "PERMIT_TYPE", "STRUCTURE_TYPE", "WORK",
    "STATUS", "CURRENT_USE", "PROPOSED_USE", "DESCRIPTION",
    "POSTAL", "fsa", "WARD_GRID", "GEO_ID",
    "APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE",
    "app_year", "app_month", "app_year_month", "approval_days",
    "est_cost", "DWELLING_UNITS_CREATED", "DWELLING_UNITS_LOST",
    "sector", "work_category", "source",
]
missing = [c for c in keep_cols if c not in permits.columns]
print("Columns not found (check names):", missing)

clean = permits[[c for c in keep_cols if c in permits.columns]]
out = CLEAN / "permits_clean.csv"
clean.to_csv(out, index=False)
print(f"Saved {len(clean):,} rows and {clean.shape[1]} columns to {out}")

In [ ]:
# Paste this table into docs/cleaning_log.md
log_df = pd.DataFrame(cleaning_log)
try:
    print(log_df.to_markdown(index=False))  # needs the optional "tabulate" package
except ImportError:
    print(log_df.to_string(index=False))